In [1]:
import pandas as pd
import pickle

df = pd.read_pickle('../output/df_vektorisiert.pkl')
with open('../output/vektoren.pkl', 'rb') as f:
    v = pickle.load(f)
tfidf, tfidf_matrix = v['tfidf'], v['tfidf_matrix']
count_vectorizer, count_matrix = v['count_vectorizer'], v['count_matrix']

In [2]:
from sklearn.decomposition import LatentDirichletAllocation, NMF
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

tokenized_texts = df['processed_tokens'].tolist()
dictionary = Dictionary(tokenized_texts)

def coherence_lda(k):
    model = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=20, learning_method='batch')
    model.fit(count_matrix)
    feature_names = count_vectorizer.get_feature_names_out()
    topics = [[feature_names[i] for i in topic.argsort()[-10:][::-1]] for topic in model.components_]
    cm = CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=dictionary, coherence='c_v')
    return model, cm.get_coherence()

def coherence_nmf(k):
    model = NMF(n_components=k, random_state=42, init='nndsvd', max_iter=300)
    model.fit(tfidf_matrix)
    feature_names = tfidf.get_feature_names_out()
    topics = [[feature_names[i] for i in topic.argsort()[-10:][::-1]] for topic in model.components_]
    cm = CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=dictionary, coherence='c_v')
    return model, cm.get_coherence()

k_werte = [5, 8, 10, 12, 15, 20]
print("Teste verschiedene Themenanzahlen, das dauert je nach Rechner einige Minuten...")
ergebnisse_lda = {k: coherence_lda(k) for k in k_werte}
ergebnisse_nmf = {k: coherence_nmf(k) for k in k_werte}

for k in k_werte:
    print(f"k={k:>2} | LDA c_v={ergebnisse_lda[k][1]:.4f} | NMF c_v={ergebnisse_nmf[k][1]:.4f}")

Teste verschiedene Themenanzahlen, das dauert je nach Rechner einige Minuten...
k= 5 | LDA c_v=0.3959 | NMF c_v=0.4541
k= 8 | LDA c_v=0.3984 | NMF c_v=0.4075
k=10 | LDA c_v=0.3924 | NMF c_v=0.4210
k=12 | LDA c_v=0.3677 | NMF c_v=0.4178
k=15 | LDA c_v=0.3546 | NMF c_v=0.4090
k=20 | LDA c_v=0.3571 | NMF c_v=0.4039


In [3]:
bestes_nmf_k = max(ergebnisse_nmf, key=lambda k: ergebnisse_nmf[k][1])
print(f"Bestes k für NMF laut Coherence Score: {bestes_nmf_k}")

nmf_final, _ = ergebnisse_nmf[bestes_nmf_k]
nmf_verteilung = nmf_final.transform(tfidf_matrix)
df['nmf_dominant_topic'] = nmf_verteilung.argmax(axis=1)

kreuztabelle = pd.crosstab(df['category_parent'], df['nmf_dominant_topic'])
print(kreuztabelle)

Bestes k für NMF laut Coherence Score: 5
nmf_dominant_topic             0    1    2    3    4
category_parent                                     
Beleuchtung                  142   90  193   52  140
Hinweise                       1   14    2   27    0
Muell/Sauberkeit              16   84    9  507    0
Oeffentliche Orte/Ufer         4   38   16  108    2
Pflanzenwuchs                  2   41    1  132    0
Strassen, Fahrrad & Verkehr    9  194   34  668   14


In [4]:
df.to_pickle('../output/df_themen.pkl')
with open('../output/modelle.pkl', 'wb') as f:
    pickle.dump({'ergebnisse_lda': ergebnisse_lda, 'ergebnisse_nmf': ergebnisse_nmf, 'bestes_nmf_k': bestes_nmf_k}, f)

In [5]:
k_test = 10
nmf_test, _ = ergebnisse_nmf[k_test]
verteilung_test = nmf_test.transform(tfidf_matrix)
df[f'nmf_topic_k{k_test}'] = verteilung_test.argmax(axis=1)
print(pd.crosstab(df['category_parent'], df[f'nmf_topic_k{k_test}']))

nmf_topic_k10                 0   1    2    3    4    5    6   7    8    9
category_parent                                                           
Beleuchtung                  50  42  151    0  102   20   44  87    6  115
Hinweise                      0   9    2    0    0    1   27   0    5    0
Muell/Sauberkeit             11  46    4  167    0   77  148   5  155    3
Oeffentliche Orte/Ufer        4  21   17    9    0    9   90   0   18    0
Pflanzenwuchs                 2  21    0    0    0    3  138   1   11    0
Strassen, Fahrrad & Verkehr   7  90   22    4    5  282  421   4   79    5


In [6]:
df.to_pickle('../output/df_themen.pkl')
with open('../output/modelle.pkl', 'wb') as f:
    pickle.dump({'ergebnisse_lda': ergebnisse_lda, 'ergebnisse_nmf': ergebnisse_nmf, 'bestes_nmf_k': bestes_nmf_k}, f)
    